# 02 — Preprocessing Validation (Stage 4 / Stage 5)

Confirms normalization, patch extraction, and the DataLoader/augmentation pipeline behave as expected before any model training.

In [ ]:
import sys; sys.path.append('/kaggle/working/SAR2EO-Diff')
from src.data.dataset import SEN12MSPairedDataset
from src.data.transforms import PairedAugment
from torch.utils.data import DataLoader
import torch

## Build train/val DataLoaders on a small subset

In [ ]:
DATASET_ROOT = '/kaggle/input/sen12ms'

train_ds = SEN12MSPairedDataset(root=DATASET_ROOT, split='train', patch_size=128,
                                 transform=PairedAugment(), subset_size=50)
val_ds = SEN12MSPairedDataset(root=DATASET_ROOT, split='val', patch_size=128, subset_size=50)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print('Batch SAR shape:', batch['sar'].shape)
print('Batch EO shape:', batch['eo'].shape)

## Verify augmentation keeps SAR/EO spatially aligned

Visually confirm a flipped/rotated SAR patch and its EO pair still correspond to the same geographic content (e.g. a distinctive field boundary should appear in the same place in both).

In [ ]:
import matplotlib.pyplot as plt
from src.data.preprocessing import denormalize_sar, denormalize_eo
import numpy as np

sar_np = batch['sar'][0].numpy()
eo_np = denormalize_eo(batch['eo'][0].numpy())

fig, axes = plt.subplots(1, 2, figsize=(8,4))
axes[0].imshow(denormalize_sar(sar_np[0]), cmap='gray'); axes[0].set_title('SAR VV (augmented)')
axes[1].imshow(np.transpose(eo_np, (1,2,0))); axes[1].set_title('EO RGB (augmented, same sample)')
for a in axes: a.axis('off')
plt.show()

## Timing sanity check

Measure DataLoader throughput so you can estimate epoch time before committing to a full training run on limited Kaggle GPU hours.

In [ ]:
import time
start = time.time()
for i, batch in enumerate(train_loader):
    if i >= 10:
        break
elapsed = time.time() - start
print(f'{elapsed:.2f}s for {i+1} batches -> {elapsed/(i+1):.3f}s/batch')